# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant data packaging standard.

### Dataset Source
The FAIR² dataset is defined by a Croissant schema available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let us inspect the available record sets and their fields and IDs (`@id`).

Below, we enumerate the record sets, their fields, and the corresponding field and record set `@id` values. You’ll need these IDs for precise access throughout your analysis.

In [ ]:
# List all record sets, their @id, and their fields (with @id)

def get_entity_id(entity):
    # Handles both dict or Croissant's internal classes
    if hasattr(entity, '@id'):
        return getattr(entity, '@id')
    elif isinstance(entity, dict) and '@id' in entity:
        return entity['@id']
    elif hasattr(entity, 'id'):
        return getattr(entity, 'id')
    return None

record_set_ids = []
print("\nAvailable record sets in the dataset (with their '@id'):\n")
for record_set in dataset.record_sets:
    rs_id = get_entity_id(record_set)
    rs_name = getattr(record_set, 'name', None) or rs_id
    print(f"- {rs_name}\n  @id: {rs_id}")
    record_set_ids.append(rs_id)
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            fname = getattr(field, 'name', None) or get_entity_id(field)
            print(f"    - Field: {fname}  (@id: {get_entity_id(field)})")
    print()
if not record_set_ids:
    print("No record sets found in the schema. The dataset may provide only one tabular set via 'record_sets', which could be loaded directly. We attempt to list records from the main record set below.")

## 3. Data Extraction
We now extract the data from each record set for further analysis. Specify your desired record set(s) and field `@id`s from the overview above.

In [ ]:
# If record set IDs were found, load them; otherwise, try to load default records.
if record_set_ids:
    record_sets_to_load = record_set_ids
else:
    # fallback: try to list all records without specifying record_set
    record_sets_to_load = [None]

dataframes = {}
for rs_id in record_sets_to_load:
    try:
        if rs_id is not None:
            records = list(dataset.records(record_set=rs_id))
        else:
            records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes[rs_id or 'default'] = df
        print(f"Loaded {len(df)} records from record set @id={rs_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Error loading record set {rs_id}: {e}")

# For the rest of this notebook, we use the primary record set.
if record_set_ids:
    primary_rs_id = record_set_ids[0]
else:
    primary_rs_id = 'default'

df = dataframes[primary_rs_id]


## 4. Exploratory Data Analysis (EDA)
This section demonstrates:
- Filtering by a numeric field (e.g., patient age or interval)
- Normalizing the field
- Grouping by a categorical field (e.g., anatomical location)

**Note:** In all steps, we reference fields by their `@id`. Adjust `numeric_field_id` and `group_field_id` as needed based on the printed columns above.

In [ ]:
# Example: Let's try 'cr:Age' and 'cr:AnatomicalLocation' as field @id if present.
possible_numeric_fields = ['cr:Age', 'cr:IntervalMonths', 'cr:IntervalYears', 'cr:DiagnosisInterval']
found_numeric_field = None

# Inspect columns to select first candidate numeric field
for col in df.columns:
    if col in possible_numeric_fields:
        found_numeric_field = col
        break

# If none matched, just pick the first float/int-typed column
if found_numeric_field is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            found_numeric_field = col
            break

if found_numeric_field:
    numeric_field_id = found_numeric_field
    print(f"Using numeric field @id: {numeric_field_id}")
    # Try to choose threshold automatically
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    threshold = float(threshold)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.1f} (mean):")
    print(filtered_df.head())

    # Normalization
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No suitable numeric field found for EDA.")

# Now choose a group field (categorical such as anatomical location, sex, etc.)
possible_group_fields = ['cr:AnatomicalLocation', 'cr:Sex', 'cr:MSIStatus', 'cr:HistopathologicalSubtype']
found_group_field = None
for col in df.columns:
    if col in possible_group_fields:
        found_group_field = col
        break

if found_numeric_field and found_group_field:
    group_field_id = found_group_field
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    print(f"\nGrouped filtered data by {group_field_id} (@id):")
    print(grouped_df)
else:
    print("No appropriate group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields. Below, we generate a histogram of the selected numeric field and a box plot by group if a grouping field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')

if found_numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if found_group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field found to visualize.")

## 6. Conclusion
In this notebook, you have learned how to use the `mlcroissant` library to:
- Load dataset metadata and tabular data from a Croissant schema-defined FAIR² dataset
- Systematically reference all record sets and fields by their `@id`
- Perform exploratory data analysis: filter, normalize, and group by relevant fields
- Visualize dataset characteristics

The FAIR² dataset enables further clinical and statistical analysis of second primary colorectal cancer in survivors. You may extend this notebook by enriching visualizations, performing statistical testing, and modeling as appropriate for your downstream tasks.